# 02 · Preprocesamiento y selección de variables

**Fase CRISP-DM:** preparación de los datos.

1. Verificar que no existan identificadores (IP, puertos, marcas de tiempo).
2. Separar entrenamiento y validación, manteniendo intacta la prueba oficial.
3. Seleccionar variables usando solo el conjunto de entrenamiento.
4. Revisar cuánto separa cada variable las clases por sí sola.
5. Definir el tratamiento del desbalance.
6. Guardar `train`, `val`, `test` y `variables.json` en `data/processed/`.

In [ ]:
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ.resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.utils.class_weight import compute_class_weight

from src.data_loader import (
    CLASES, cargar_procesado, columnas_caracteristicas,
    guardar_procesado, guardar_variables,
)
from src.features import VARIABLES_SOSPECHOSAS, seleccionar_variables, preprocesador

SEMILLA = 42

train = cargar_procesado("train_limpio")
test = cargar_procesado("test_limpio")
variables = columnas_caracteristicas(train)

print("Entrenamiento limpio:", train.shape)
print("Prueba oficial:", test.shape)

## 1. Identificadores sesgados

El informe indica excluir identificadores como direcciones IP y puertos,
porque el modelo podría memorizar equipos concretos en vez de aprender
el comportamiento del tráfico.

In [ ]:
identificadores_conocidos = {
    "flow id", "src ip", "dst ip", "source ip", "destination ip",
    "src port", "dst port", "source port", "destination port",
    "timestamp", "src mac", "dst mac",
}
identificadores = [c for c in variables if c.lower() in identificadores_conocidos]
print("Identificadores encontrados:", identificadores or "ninguno")

## 2. Partición de entrenamiento y validación

El 20 % del entrenamiento se reserva como **validación** para comparar
modelos y ajustar hiperparámetros. La **prueba oficial** se usa solo
para la evaluación final.

Como los archivos de prueba ya fueron examinados en la etapa anterior,
su evaluación se considera exploratoria (ver `DESCARGA_DATOS.txt`).

In [ ]:
entrenamiento, validacion = train_test_split(
    train,
    test_size=0.20,
    stratify=train["clase"],
    random_state=SEMILLA,
)

display(pd.DataFrame({
    "Entrenamiento": entrenamiento["clase"].value_counts(),
    "Validación": validacion["clase"].value_counts(),
    "Prueba oficial": test["clase"].value_counts(),
}).sort_index().rename(index=CLASES))

## 3. Selección de variables

Con el conjunto de entrenamiento se eliminan:

- variables **constantes**;
- variables con **correlación absoluta mayor al umbral** con otra ya
  conservada (por ejemplo, columnas idénticas como `Rate` y `Srate`).

`EXCLUIR_SOSPECHOSAS` permite además quitar `IAT`, `Number` y `Weight`.
Estas variables pueden reflejar cómo se generó la captura (temporización
o tamaño de ventana) más que el comportamiento del ataque. Conviene
comparar los resultados con y sin ellas.

In [ ]:
EXCLUIR_SOSPECHOSAS = False
UMBRAL_CORRELACION = 0.98

excluir = VARIABLES_SOSPECHOSAS if EXCLUIR_SOSPECHOSAS else []
seleccionadas, informe = seleccionar_variables(
    entrenamiento[variables], UMBRAL_CORRELACION, excluir
)

print(f"Variables seleccionadas: {len(seleccionadas)} de {len(variables)}")
display(informe)
print(seleccionadas)

## 4. Poder de separación de cada variable

Se entrena un árbol de profundidad 2 con **una sola variable** y se mide
la exactitud balanceada en validación. Con tres clases, el azar
corresponde a 0,33.

Si una variable sola separa casi perfectamente las clases, el problema
es sencillo dentro de este dominio. Sin embargo, el modelo puede
depender de esa variable y fallar si su distribución cambia en otra red.

In [ ]:
muestra_ent = entrenamiento.sample(
    min(len(entrenamiento), 100_000), random_state=SEMILLA
)

filas = []
for variable in seleccionadas:
    arbol = make_pipeline(
        preprocesador(),
        DecisionTreeClassifier(max_depth=2, random_state=SEMILLA),
    )
    arbol.fit(muestra_ent[[variable]], muestra_ent["clase"])
    pred = arbol.predict(validacion[[variable]])
    filas.append({
        "Variable": variable,
        "Exactitud balanceada": balanced_accuracy_score(validacion["clase"], pred),
    })

separacion = (
    pd.DataFrame(filas).set_index("Variable")
    .sort_values("Exactitud balanceada", ascending=False)
)
display(separacion.head(15).round(4))

separacion.head(15).sort_values("Exactitud balanceada").plot.barh(
    figsize=(8, 6), legend=False,
    title="Exactitud balanceada usando una sola variable"
)
plt.axvline(1 / len(CLASES), color="gray", linestyle="--", label="Azar")
plt.xlabel("Exactitud balanceada en validación")
plt.legend()
plt.tight_layout()
plt.show()

## 5. Desbalance de clases

Se usan **pesos por clase** (`class_weight="balanced"` o pesos por
muestra equivalentes) en lugar de sobremuestreo sintético como SMOTE:

- el conjunto es grande y todas las clases tienen muchos registros;
- generar flujos sintéticos puede producir combinaciones de variables
  que no existen en tráfico real;
- los pesos no modifican los datos y se aplican igual a todos los modelos.

In [ ]:
def pesos(y):
    clases = np.unique(y)
    valores = compute_class_weight("balanced", classes=clases, y=y)
    return {int(c): round(float(v), 4) for c, v in zip(clases, valores)}

pesos_binaria = pesos(entrenamiento["etiqueta"])
pesos_multiclase = pesos(entrenamiento["clase"])

print("Pesos binaria (0 = benigno, 1 = DDoS):", pesos_binaria)
print("Pesos multiclase:", {CLASES[c]: p for c, p in pesos_multiclase.items()})

## 6. Guardar los conjuntos procesados

Se guardan solo las variables seleccionadas y las dos etiquetas. La
imputación (y el escalado en modelos lineales) se ajusta luego dentro
de cada pipeline, usando solo entrenamiento.

In [ ]:
columnas = seleccionadas + ["clase", "etiqueta"]
conjuntos = {"train": entrenamiento, "val": validacion, "test": test}

for nombre, datos in conjuntos.items():
    ruta = guardar_procesado(datos[columnas].reset_index(drop=True), nombre)
    print(f"{nombre}: {datos.shape[0]} registros → {ruta}")

config = {
    "variables": seleccionadas,
    "excluidas": informe["Motivo"].to_dict(),
    "excluir_sospechosas": EXCLUIR_SOSPECHOSAS,
    "umbral_correlacion": UMBRAL_CORRELACION,
    "pesos_binaria": pesos_binaria,
    "pesos_multiclase": pesos_multiclase,
    "semilla": SEMILLA,
}
print("Configuración:", guardar_variables(config))

## Conclusiones del preprocesamiento

Completar con los resultados obtenidos:

1. Variables eliminadas y motivo.
2. Variables que por sí solas separan las clases (posibles atajos).
3. Tamaño final de entrenamiento, validación y prueba oficial.